In [ ]:
# --- imports 
import polars as pl
import pandas as pd
import numpy as np
import os
import requests
import logging

# logging configuration
logging.basicConfig(level=logging.INFO)

In [ ]:
# --- constants 

USED_NODE: str = "thin002"
T_START:   str = "2026-08-11T00:00:00.000000Z"
T_END:     str = "2026-08-11T23:59:59.000000Z"
DATA_DIR:  str = "/Users/isacpasianotto/Desktop/workdirectoy/paper-enegy/data-analysis/data"
SENSOR_FILE: str = DATA_DIR + "/ipmi_sensors.arrow"
POWER_FILE:  str = DATA_DIR + "/ipmi_power.arrow"
QUESTDB_ENDPOINT: str = "https://timeseriesdb.dev.rd.areasciencepark.it"


QUERY_SENSORS = f"""
SELECT *
FROM ipmi_sensor
WHERE host LIKE '{USED_NODE}%'
AND timestamp >= '{T_START}'
AND timestamp <= '{T_END}'
ORDER BY timestamp ASC;
"""

QUERY_POWER = f"""
SELECT *
FROM ipmi_power
WHERE host LIKE '{USED_NODE}%'
AND timestamp >= '{T_START}'
AND timestamp <= '{T_END}'
ORDER BY timestamp ASC;
"""

In [ ]:
def get_table_data(filename: str, query: str) -> pl.DataFrame:
    """
    Check if `filename` exists, if so try to read it. 
    Otherwise, query the QuestDB via REST API and save the result to `filename`.
    """
    
    if os.path.exists(filename):
        try:
            logging.info(f"Reading data from {filename}")
            return pl.read_ipc(filename)
        except Exception as e:
            print(f"Error reading {filename}: {e}")
            return pl.DataFrame()  # return empty DataFrame on error

    # --- query a QuestDB via REST API ---
    logging.info(f"Querying QuestDB and saving to {filename}")
    
    response = requests.get(
        f"{QUESTDB_ENDPOINT}/exec",
        params={"query": query},
        verify=False  # Disable SSL verification for self-signed certificates
    )
    response.raise_for_status()
    payload = response.json()

    columns = [c["name"] for c in payload["columns"]]
    rows = payload["dataset"]

    df = pl.DataFrame(rows, schema=columns, orient="row")

    # --- convert columns to appropriate types ---
    logging.debug(f"Converting columns to appropriate types for {filename}")
    df = df.with_columns(
        pl.col("timestamp").str.strptime(
            pl.Datetime, "%Y-%m-%dT%H:%M:%S%.fZ", strict=False
        ),
        pl.col("value").cast(pl.Float64),
    )
    for col in ("name", "type", "unit"):
        if col in df.columns:
            df = df.with_columns(pl.col(col).cast(pl.Utf8))

    if "psu_id" in df.columns:
        df = df.with_columns(pl.col("psu_id").cast(pl.Int64, strict=False))

    os.makedirs(os.path.dirname(filename), exist_ok=True)
    df.write_ipc(filename)

    return df

In [ ]:
power_df: pl.DataFrame = get_table_data(POWER_FILE, QUERY_POWER)
sensor_df: pl.DataFrame = get_table_data(SENSOR_FILE, QUERY_SENSORS)

## Data-cleaning 

***Step 1:*** Reshape the 2 dataframe keeping only the data of interests, and merge them into a single one

In [ ]:
power_reshaped: pl.DataFrame = (
    power_df
    .filter(
        (pl.col("unit") == "W") &
        (pl.col("psu_id") == 0)
    )
    .select(
        "timestamp",
        pl.col("value").alias("power_W")
    )
    .drop_nulls()
)

In [ ]:
# print(sensor_df.head())

sensors_filtered = pl.sql("""
    SELECT host, name, value, timestamp
    FROM sensor_df
    WHERE unit IN ('C', 'RPM')
""").collect()

sensors_reshaped: pl.DataFrame = sensors_filtered.pivot(
    on="name",
    index=["host", "timestamp"],
    values="value",
    ).drop("host").drop_nulls().sort("timestamp")

Merge the two Dataframe on the `timestamp` column.

In [ ]:
# NOTE --> `join_asof` merge the df on a column with the closest value, while merge 
#           checks for exact matches. This is useful with time series :)

merged_df: pl.DataFrame = power_reshaped.sort("timestamp").join_asof(
    sensors_reshaped.sort("timestamp"),
    on="timestamp",
    strategy="nearest", # or "backward" or "forward"
    tolerance="500m",
).drop_nulls()

In [ ]:
print(merged_df.head())

---

# 2. Modelling the fan contribution to node power

## 2.1 What the experiment measured

During the acquisition window the node `thin002` (a Dell PowerEdge R640, 8 dual-rotor
hot-swap fan modules) was **otherwise idle**: no user jobs, no batch workload, only the
operating system and the monitoring container. The only thing that was deliberately
varied is the **fan duty cycle**, driven out-of-band through the BMC

```
ipmitool raw 0x30 0x30 0x01 0x00           # disable automatic (thermal) fan control
ipmitool raw 0x30 0x30 0x02 0xff <duty>    # set ALL fans to <duty> %
```

stepping the duty from **100 % down to 25 % in steps of 5 %**, holding each setpoint for
three minutes so that both the fan speeds and the power reading reach steady state.

Two families of signals were recorded at 1 Hz and are now joined in `merged_df`:

| signal | source | what it is |
|---|---|---|
| `power_W` | `ipmi-oem dell get-instantaneous-power-consumption-data 0` | **cumulative AC input power of the whole node** (PSU id 0 is the aggregate, not a single PSU), 1 W resolution |
| `System_Board_FanNX` | `ipmi-sensors` | speed of rotor *X* ∈ {A, B} of fan module *N* ∈ {1..8}, in RPM (120 RPM quantisation) |
| `*_Temp` | `ipmi-sensors` | CPU 1/2 die, board inlet and board exhaust temperatures, in °C (1 °C quantisation) |

Note that we use the *PSU-level* power reading rather than the `System_Board_Pwr_Consumption`
SDR sensor: the latter is quantised to 24 W steps on this platform, which is coarser than the
entire effect we are trying to resolve.

## 2.2 The quantity we want

The target is an **additive, per-rotor decomposition of node power**

$$P(t) \;=\; P_0 \;+\; \sum_{i=1}^{16} m\!\left(N_i(t)\right) \;+\; \varepsilon(t), \qquad m(0)=0$$

where $N_i$ is the speed of rotor $i$, $m(\cdot)$ is the per-rotor power function and $P_0$
collects everything that is not the fans (CPU idle, DRAM refresh, chipset, NICs, BMC, PSU
conversion losses). Everything below is about (a) choosing $m$ on physical grounds,
(b) checking that it is *identifiable* from this experiment, and (c) quantifying how well it
reproduces the measured power.

Strictly, the equation above is a **power** balance (W). The energy over a window
$[t_0,t_1]$ follows by integration, and inherits the same additive structure:

$$E \;=\; \int_{t_0}^{t_1}\! P\,\mathrm{d}t \;=\; P_0\,(t_1-t_0) \;+\; \sum_{i=1}^{16}\int_{t_0}^{t_1}\! m\!\left(N_i(t)\right)\mathrm{d}t .$$

In [ ]:
# --- imports and plotting setup for the modelling section
import json
import matplotlib as mpl
import matplotlib.pyplot as plt

# Fixed categorical slots (assigned by entity, never cycled) + chart chrome, light surface.
BLUE, ORANGE, AQUA = "#2a78d6", "#eb6834", "#1baf7a"
INK, INK2, MUTED = "#0b0b0b", "#52514e", "#898781"
GRID, AXIS, SURFACE = "#e1e0d9", "#c3c2b7", "#fcfcfb"
CRITICAL, WARNING = "#d03b3b", "#fab219"

mpl.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
    "figure.dpi": 110, "savefig.dpi": 200, "font.size": 9,
    "axes.edgecolor": AXIS, "axes.linewidth": 0.8, "axes.labelcolor": INK2,
    "axes.titlesize": 10, "axes.titleweight": "bold", "axes.titlecolor": INK,
    "axes.titlelocation": "left", "axes.titlepad": 8,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "axes.axisbelow": True,
    "grid.color": GRID, "grid.linewidth": 0.7, "grid.linestyle": "-",
    "xtick.color": MUTED, "ytick.color": MUTED,
    "xtick.labelcolor": INK2, "ytick.labelcolor": INK2,
    "legend.frameon": False, "legend.fontsize": 8.5,
    "lines.linewidth": 1.8, "lines.markersize": 5,
})

FIG_DIR = os.path.join(os.path.dirname(DATA_DIR), "figures")
os.makedirs(FIG_DIR, exist_ok=True)


def save_fig(fig, name: str) -> None:
    """Save a figure as PDF (for the paper) and PNG (for quick viewing)."""
    for ext in ("pdf", "png"):
        fig.savefig(os.path.join(FIG_DIR, f"{name}.{ext}"), bbox_inches="tight")

In [ ]:
# --- the modelling frame: rotor speeds, power, temperatures, elapsed time

FAN_COLS = [c for c in merged_df.columns if "Fan" in c]
TEMP_COLS = [c for c in merged_df.columns if "Temp" in c]
N_ROTORS = len(FAN_COLS)

model_df: pl.DataFrame = (
    merged_df
    .with_columns(
        ((pl.col("timestamp") - pl.col("timestamp").min())
         .dt.total_milliseconds() / 1000.0).alias("t_s"),
        pl.mean_horizontal(FAN_COLS).alias("rpm_mean"),
    )
)

t = model_df["t_s"].to_numpy()              # elapsed time [s]
P = model_df["power_W"].to_numpy()          # node AC input power [W]
N = model_df.select(FAN_COLS).to_numpy()    # (n, 16) rotor speeds [RPM]
K = N / 1000.0                              # rotor speeds in kRPM -> better-conditioned design
rpm_mean = N.mean(axis=1)
cpu_temp = model_df.select(["Processor_1_Temp", "Processor_2_Temp"]).to_numpy().mean(axis=1)
exhaust_temp = model_df["System_Board_Exhaust_Temp"].to_numpy()
inlet_temp = model_df["System_Board_Inlet_Temp"].to_numpy()

print(f"{len(t)} samples over {t[-1] / 60:.1f} min, median cadence "
      f"{np.median(np.diff(t)):.2f} s, {N_ROTORS} rotors")
print(f"power  : {P.min():.0f} .. {P.max():.0f} W")
print(f"rotors : {N.min():.0f} .. {N.max():.0f} RPM")
print(f"inlet  : {np.unique(inlet_temp)} C   <-- constant: no ambient variation in this run")

## 2.3 Recovering the experimental design from the trace

The fan setpoint was applied out-of-band and logged to a CSV *on the node*, not to the
time-series database, so the duty cycle is not a column we can join. We recover it from the
rotor-speed trace itself, which is piecewise constant by construction.

The detector is deliberately simple and assumption-free: a sample belongs to a **steady-state
plateau** if the standard deviation of the 16-rotor mean speed over a ±15 s window is below
60 RPM (within-plateau jitter of that mean is ~20 RPM, while the smallest step between
consecutive setpoints is ~800 RPM), and a run of such samples is kept only if it lasts at
least 60 s. Everything else — the spin-up/spin-down ramps — is excluded from *fitting* but
retained for out-of-sample evaluation later.

Working on plateau means rather than raw samples is not cosmetic: it is what makes the
statistics honest. Each three-minute plateau is **one experimental condition**, not 180
independent observations (see §3.5).

In [ ]:
STEADY_WIN = 15      # +/- samples (~ +/- 15 s at 1 Hz)
STEADY_TOL = 60.0    # RPM; within-plateau jitter of the 16-rotor mean is ~20 RPM
MIN_PLATEAU = 60     # samples; a real setpoint is held for 180 s


def steady_state_labels(signal: np.ndarray,
                        win: int = STEADY_WIN,
                        tol: float = STEADY_TOL,
                        min_len: int = MIN_PLATEAU) -> tuple[np.ndarray, np.ndarray]:
    """
    Label contiguous steady-state plateaus of a piecewise-constant signal.

    Returns (labels, rolling_sd); labels are 0, 1, 2, ... in time order, -1 for
    samples that belong to a transient (or to a run too short to be a setpoint).
    """
    n = signal.size
    rolling_sd = np.array([signal[max(0, i - win): i + win + 1].std() for i in range(n)])
    steady = rolling_sd < tol

    labels = np.full(n, -1, dtype=int)
    plateau, i = 0, 0
    while i < n:
        if not steady[i]:
            i += 1
            continue
        j = i
        while j < n and steady[j]:
            j += 1
        if j - i >= min_len:
            labels[i:j] = plateau
            plateau += 1
        i = j
    return labels, rolling_sd


plateau_id, rolling_sd = steady_state_labels(rpm_mean)
n_plateaus = int(plateau_id.max()) + 1

# The experiment is: [idle] 100, 95, ..., 25 [100 again] [idle], i.e. 16 commanded
# setpoints bracketed by three extra plateaus. Fail loudly if the trace disagrees.
SWEEP_DUTY = np.arange(100, 20, -5)              # 100, 95, ..., 25  -> 16 setpoints
PRE_ID, SWEEP_IDS, REPEAT_ID, AUTO_ID = 0, np.arange(1, 17), 17, 18
assert n_plateaus == 19, f"expected 19 plateaus (1 pre + 16 setpoints + 1 repeat + 1 post), got {n_plateaus}"
assert len(SWEEP_DUTY) == 16

duty = np.full(len(t), np.nan)                   # nominal duty cycle [%], NaN outside the sweep
for g, d in zip(SWEEP_IDS, SWEEP_DUTY):
    duty[plateau_id == g] = d

print(f"{n_plateaus} plateaus detected; "
      f"{(plateau_id >= 0).sum()} / {len(t)} samples ({100 * (plateau_id >= 0).mean():.0f} %) are steady state")

In [ ]:
# --- plateau summary table
labels_txt = {PRE_ID: "pre-sweep (auto)", REPEAT_ID: "100 % repeat", AUTO_ID: "post-sweep (auto)"}
rows = []
for g in range(n_plateaus):
    m_g = plateau_id == g
    role = labels_txt[g] if g in labels_txt else f"setpoint {int(duty[m_g][0])} %"
    rows.append({
        "plateau": g,
        "role": role,
        "t_start_s": round(float(t[m_g][0])),
        "duration_s": round(float(t[m_g][-1] - t[m_g][0])),
        "n": int(m_g.sum()),
        "rpm_mean": round(float(N[m_g].mean()), 1),
        "rpm_sd": round(float(rpm_mean[m_g].std()), 1),
        "power_W": round(float(P[m_g].mean()), 2),
        "power_sd_W": round(float(P[m_g].std()), 2),
        "cpu_C": round(float(cpu_temp[m_g].mean()), 1),
        "exhaust_C": round(float(exhaust_temp[m_g].mean()), 1),
    })
plateau_df = pl.DataFrame(rows)
with pl.Config(tbl_rows=25, tbl_width_chars=160):
    print(plateau_df)

In [ ]:
# --- Figure 1: the experiment as recorded
fig, (ax_p, ax_n) = plt.subplots(2, 1, figsize=(9.5, 5.4), sharex=True,
                                 gridspec_kw={"height_ratios": [1, 1], "hspace": 0.18})

for g in range(n_plateaus):
    m_g = plateau_id == g
    for ax in (ax_p, ax_n):
        ax.axvspan(t[m_g][0] / 60, t[m_g][-1] / 60, color=GRID, alpha=0.55, lw=0, zorder=0)

ax_p.plot(t / 60, P, color=BLUE, lw=1.1)
ax_p.set_ylabel("node power  [W]")
ax_p.set_title("Fan sweep on thin002 — shaded bands are detected steady-state plateaus")

ax_n.plot(t / 60, rpm_mean / 1000, color=INK2, lw=1.1)
ax_n.set_ylabel("mean rotor speed  [kRPM]")
ax_n.set_xlabel("elapsed time  [min]")

# selective direct labels only: the two endpoints of the sweep and the two anomalies
for g, txt, ax in ((SWEEP_IDS[0], "100 %", ax_n), (SWEEP_IDS[-1], "25 %", ax_n),
                   (REPEAT_ID, "100 % repeat", ax_n), (AUTO_ID, "auto", ax_n)):
    m_g = plateau_id == g
    ax.annotate(txt, (t[m_g].mean() / 60, rpm_mean[m_g].mean() / 1000),
                textcoords="offset points", xytext=(0, 8), ha="center",
                fontsize=8, color=INK2)
ax_p.annotate("redundant PSU leaves\nhot-spare standby (§2.4)", (277 / 60, 240),
              textcoords="offset points", xytext=(14, 22), fontsize=8, color=CRITICAL,
              arrowprops=dict(arrowstyle="-", color=CRITICAL, lw=0.9))
ax_p.annotate("unexplained +6 W step (§4.4)", (1820 / 60, 271),
              textcoords="offset points", xytext=(-30, -34), fontsize=8, color=CRITICAL,
              arrowprops=dict(arrowstyle="-", color=CRITICAL, lw=0.9))
save_fig(fig, "fig1-timeline")
plt.show()

## 2.4 Two things the trace reveals before any modelling

### The redundant PSU leaves standby when the fans spin up

For the first ~277 s the node draws **238 W**; at the *same* rotor speed at the end of the run
it draws **252 W**. The explanation is in the per-PSU readings: this chassis runs Dell's
*PSU hot-spare* policy, keeping the redundant supply in standby (≈5 W) while the active one
carries the whole load. Commanding the fans to 100 % pushes the demand past the wake
threshold, both supplies come online and share the load — and, once awake, the spare never
goes back to standby for the rest of the run.

This is a **regime change in the power-delivery path, not in the load**: two PSUs each at
~50 % of the load run at a worse point on their efficiency curve than one at ~100 %, and the
second supply adds its own standby overhead. The pre-sweep block therefore belongs to a
different system and is **excluded from the fit**; we keep it as a diagnostic and quantify the
offset in §4.4. It is a nice incidental result: hot-spare is worth ~14 W (≈5.5 %) at idle on
this node.

### An unexplained persistent +6 W step at t ≈ 1820 s

In the middle of the 60 % plateau the power rises by ~6 W with no change in rotor speed and no
change in any temperature, and stays there. It is visible in both PSU readings, so it is a
genuine load change on the node, not a sensor artefact — most likely a background system
activity that started and did not stop. We do not model it away; we let the residual analysis
in §4.4 expose it and quantify what it costs the fit.

In [ ]:
# --- per-PSU decomposition of the node power (psu_id 0 is the cumulative reading)
psu_df: pl.DataFrame = (
    power_df
    .filter(pl.col("unit") == "W")
    .pivot(on="psu_id", index="timestamp", values="value")
    .sort("timestamp")
    .drop_nulls()
)
psu_t = ((psu_df["timestamp"] - model_df["timestamp"].min()).dt.total_milliseconds() / 1000.0).to_numpy()
psu_1, psu_2 = psu_df["1"].to_numpy(), psu_df["2"].to_numpy()

wake = psu_t[np.argmax(psu_2 > 20)]
pre = (plateau_id == PRE_ID)
post = (plateau_id == AUTO_ID)
print(f"redundant PSU leaves standby at t = {wake:.0f} s")
print(f"idle power, 1 PSU active (pre-sweep) : {P[pre].mean():6.2f} W  at {rpm_mean[pre].mean():.0f} RPM")
print(f"idle power, 2 PSUs active (post-sweep): {P[post].mean():6.2f} W  at {rpm_mean[post].mean():.0f} RPM")
print(f"hot-spare saving                      : {P[post].mean() - P[pre].mean():6.2f} W "
      f"({100 * (P[post].mean() - P[pre].mean()) / P[post].mean():.1f} % of node power)")

fig = plt.figure(figsize=(9.5, 3.6))
gs = fig.add_gridspec(2, 2, width_ratios=[1.15, 1], hspace=0.16, wspace=0.3)
ax_a = fig.add_subplot(gs[:, 0])
ax_b1 = fig.add_subplot(gs[0, 1])
ax_b2 = fig.add_subplot(gs[1, 1], sharex=ax_b1)

ax_a.plot(psu_t / 60, psu_1, color=BLUE, lw=1.1, label="PSU 1")
ax_a.plot(psu_t / 60, psu_2, color=ORANGE, lw=1.1, label="PSU 2 (hot spare)")
ax_a.axvline(wake / 60, color=CRITICAL, lw=0.9)
ax_a.annotate("spare wakes", (wake / 60, 210), textcoords="offset points", xytext=(8, 0),
              fontsize=8, color=CRITICAL)
ax_a.set(xlabel="elapsed time  [min]", ylabel="PSU input power  [W]")
ax_a.set_title("a · load sharing between supplies")
ax_a.legend(loc="upper right")

# the point of panel b: power moves while rotor speed does not
zoom = (t > 1700) & (t < 1950)
for ax, y, lbl in ((ax_b1, P, "node power  [W]"), (ax_b2, rpm_mean / 1000, "mean rotor\nspeed [kRPM]")):
    ax.plot(t[zoom] / 60, y[zoom], color=BLUE, lw=1.2)
    ax.axvline(1820 / 60, color=CRITICAL, lw=0.9)
    ax.set_ylabel(lbl, fontsize=8)
ax_b1.tick_params(labelbottom=False)
ax_b1.annotate("+6 W here…", (1820 / 60, 273.5), textcoords="offset points", xytext=(8, 0),
               fontsize=8, color=CRITICAL)
ax_b2.annotate("…with the fans untouched", (1820 / 60, 11.42), textcoords="offset points",
               xytext=(8, 0), fontsize=8, color=CRITICAL)
ax_b2.set_xlabel("elapsed time  [min]")
ax_b1.set_title("b · +6 W with no change in fan speed")
save_fig(fig, "fig2-psu-regime")
plt.show()

## 2.5 The 16 rotors carry one degree of freedom

The fans were commanded with `0xff` — *all fans* — so every rotor speed is a deterministic
function of a single scalar duty cycle $u$:

$$N_i(t) \;\approx\; a_i \, u(t) .$$

The consequence is severe and needs to be stated before, not after, fitting: the 16 rotor
columns are **collinear to numerical precision**. Below we check it three ways — the pairwise
correlations, the condition number of the unrestricted design matrix, and the visual collapse
of all 16 speed curves onto one after normalising by each rotor's own maximum. §3.3 explains
what this does and does not prevent.

In [ ]:
# --- rotor scaling: fit N_i = a_i * mean speed + b_i on the plateau means
plateau_N = np.array([N[plateau_id == g].mean(axis=0) for g in range(n_plateaus)])
plateau_mean_rpm = plateau_N.mean(axis=1)

rows = []
for i, col in enumerate(FAN_COLS):
    A = np.c_[plateau_mean_rpm, np.ones(n_plateaus)]
    coef, *_ = np.linalg.lstsq(A, plateau_N[:, i], rcond=None)
    rows.append({
        "rotor": col.replace("System_Board_", ""),
        "a_i": round(float(coef[0]), 4),
        "b_i_RPM": round(float(coef[1]), 1),
        "max_dev_RPM": round(float(np.abs(plateau_N[:, i] - A @ coef).max()), 1),
        "N_at_100pct": round(float(plateau_N[SWEEP_IDS[0], i])),
        "N_at_25pct": round(float(plateau_N[SWEEP_IDS[-1], i])),
    })
rotor_df = pl.DataFrame(rows)
with pl.Config(tbl_rows=20):
    print(rotor_df)

fit_mask = np.isin(plateau_id, SWEEP_IDS)
corr = np.corrcoef(N[fit_mask].T)
design_16 = np.c_[np.ones(fit_mask.sum()), K[fit_mask], K[fit_mask] ** 2, K[fit_mask] ** 3]
print(f"\nsmallest pairwise correlation among the 16 rotors : {corr[~np.eye(N_ROTORS, dtype=bool)].min():.4f}")
print(f"condition number of the unrestricted cubic design  : {np.linalg.cond(design_16):.3g}")
print(f"numerical rank of that design ({design_16.shape[1]} columns)         : "
      f"{np.linalg.matrix_rank(design_16)}  <- nominally full, see below")
print("\nThe design is full rank only because the 120 RPM sensor quantisation breaks exact")
print("proportionality. That residual jitter is noise, not signal: it is what an unrestricted")
print("per-rotor fit would be forced to estimate from. §3.3 shows what that produces.")

In [ ]:
# --- Figure 3: one degree of freedom
fig, (ax_a, ax_b) = plt.subplots(1, 2, figsize=(9.5, 3.4), gridspec_kw={"wspace": 0.26})

is_a_rotor = np.array([c.endswith("A") for c in FAN_COLS])
for i in range(N_ROTORS):
    ax_a.plot(SWEEP_DUTY, plateau_N[SWEEP_IDS, i] / 1000,
              color=BLUE if is_a_rotor[i] else ORANGE, lw=1.4, alpha=0.75)
ax_a.plot([], [], color=BLUE, lw=1.8, label="rotor A (front impeller)")
ax_a.plot([], [], color=ORANGE, lw=1.8, label="rotor B (rear impeller)")
ax_a.set(xlabel="commanded duty cycle  [%]", ylabel="rotor speed  [kRPM]")
ax_a.set_title("a · all 16 rotors follow one command")
ax_a.legend(loc="lower left")
ax_a.invert_xaxis()

for i in range(N_ROTORS):
    ax_b.plot(SWEEP_DUTY, plateau_N[SWEEP_IDS, i] / plateau_N[SWEEP_IDS[0], i],
              color=BLUE if is_a_rotor[i] else ORANGE, lw=1.4, alpha=0.75)
ax_b.set(xlabel="commanded duty cycle  [%]", ylabel="speed / own speed at 100 %")
ax_b.set_title("b · normalised, the 16 curves collapse onto one")
ax_b.invert_xaxis()
ax_b.annotate(f"min pairwise corr = {corr[~np.eye(N_ROTORS, dtype=bool)].min():.4f}",
              (0.04, 0.08), xycoords="axes fraction", fontsize=8, color=INK2)
save_fig(fig, "fig3-collinearity")
plt.show()

---

# 3. Theory: what functional form should $m(\cdot)$ have?

## 3.1 The fan affinity laws

Dimensional analysis of the flow through a family of geometrically similar turbomachines
yields three dimensionless groups — flow, pressure and power coefficients
(Dixon & Hall, *Fluid Mechanics and Thermodynamics of Turbomachinery*, 7th ed., 2014, ch. 1):

$$\varphi=\frac{Q}{N D^{3}},\qquad \psi=\frac{\Delta p}{\rho N^{2} D^{2}},\qquad
\lambda=\frac{P_\text{air}}{\rho N^{3} D^{5}} .$$

For **one** machine (fixed $D$) moving air of fixed density $\rho$ along a fixed system
resistance curve, the operating point stays at constant $(\varphi,\psi,\lambda)$, so varying
only the rotational speed $N$ gives the **fan affinity laws**:

$$Q \propto N, \qquad \Delta p \propto N^{2}, \qquad P_\text{air} = Q\,\Delta p \propto N^{3}.$$

The aerodynamic power of a fan therefore scales with the **cube** of its speed — the standard
result in fan engineering (Bleier, *Fan Handbook*, McGraw-Hill 1997, ch. 2; ASHRAE
*Handbook — HVAC Systems and Equipment*, ch. "Fans"; ANSI/AMCA 210). It is also the
assumption on which the server thermal-management literature is built: Wang et al.
(ASME InterPACK 2009) derive optimal fan-speed control from $P_\text{fan}\propto N^{3}$, and
it is precisely this cubic steepness that makes fan speed such a powerful control knob —
halving the speed cuts fan power by roughly a factor of eight, which is why raising data-centre
inlet temperature is a trade and not a free win (El-Sayed et al., SIGMETRICS 2012;
Patterson, ITHERM 2008; ASHRAE TC 9.9 *Thermal Guidelines*).

## 3.2 Why the *electrical* power is not exactly $N^{3}$

What IPMI reports is the **AC input power of the whole node**, not the aerodynamic power of
one impeller. Several loss mechanisms sit between the two, and they scale differently:

| mechanism | scaling in $N$ | where it dominates |
|---|---|---|
| aerodynamic (useful) work | $N^{3}$ | high speed |
| bearing friction / windage | $N$ … $N^{2}$ | low speed |
| motor copper loss $I^{2}R$ (torque $\tau\propto N^{2}$) | $N^{4}$ | high speed |
| motor core loss (hysteresis $N$, eddy $N^{2}$) | $N^{1.5}$ … $N^{2}$ | mid |
| BLDC driver, Hall sensors, tach logic | $\approx$ const | always |
| PSU conversion efficiency | non-linear in load | always (AC input $\neq$ DC load) |

The measured electrical power of a server fan is therefore a smooth, monotone, convex function
of speed that is *close to* cubic, generally with an effective exponent slightly **above** 3
once copper losses matter, plus a small offset from the driver electronics at low speed.

A **low-order polynomial in $N$** is the natural empirical form. It nests the affinity law as
its $N^{3}$ term, it lets the data say how much lower-order loss behaviour is present, and it
does not impose a functional form we cannot justify. Per rotor:

$$\boxed{\;m(N)\;=\;\beta_1 N + \beta_2 N^{2} + \beta_3 N^{3},\qquad m(0)=0\;}$$

The constraint $m(0)=0$ is physical — a stopped rotor draws no power — and it is exactly what
makes the decomposition *well posed*. Without it every rotor would contribute its own
constant, and 16 constants are indistinguishable from the baseline $P_0$.

## 3.3 The additive structure, and what "each fan alone" can mean here

Additivity over components, each component a simple function of one utilisation-like signal
fitted by least squares, is the conventional shape of full-system power models in the systems
literature — Mantis (Economou et al., MoBS 2006), the model comparison of Rivoire et al.
(HotPower 2008), and the operational models of Fan, Weber & Barroso (ISCA 2007) all take it.
We use that structure with rotor speed as the signal.

**The unrestricted version of the model is not estimable from this experiment.** Ideally we
would fit 16 separate functions $m_i$. But §2.5 showed the design carries *one* degree of
freedom: rotor speeds are proportional to a single commanded duty and pairwise correlations
exceed 0.998.

Strictly the design matrix is not *exactly* rank-deficient — the 120 RPM quantisation of the
tachometers breaks perfect proportionality, so `matrix_rank` reports full rank and least
squares returns an answer. That is precisely the trap: the only thing distinguishing one rotor
column from another is sensor noise, so the "answer" is an arbitrary point in a near-null
space, with a condition number of order $10^{6}$ (Belsley, Kuh & Welsch, *Regression
Diagnostics*, 1980, ch. 3). The cell below fits it anyway, to show concretely what comes out.
No amount of data collected *this way* fixes it; only an experiment that drives rotors
independently would.

What **is** identifiable, and what we fit, is a $m(\cdot)$ **shared** by all rotors. That is
not a resigned compromise: the eight modules are identical parts in identical slots, so
exchangeability is the physically correct restriction. And it still delivers what we want —
a *separate* power figure $m(N_i)$ for every rotor, and those figures differ, because the
rotors run at different speeds (the counter-rotating A and B impellers of a module differ by
~28 %). Only the functional *form* is shared; the per-rotor contribution is not.

## 3.4 Why temperature is left out

Excluding temperature is not just convenient here, it is the *correct* choice, for two reasons.

**Causally**: die temperature is not an independent driver of power in this experiment — it is
a *consequence* of fan speed. Conditioning on a post-treatment variable that lies on the causal
path `fan speed → cooling → die temperature` biases the coefficient of interest
(Rosenbaum, *JRSS A* 147(5), 1984). Regressing power on both would split the fan effect between
the two regressors in an arbitrary way.

**Empirically**: the inlet temperature is pinned at 20 °C for the entire run — the sensor's
1 °C quantisation records literally no variation — so there is no ambient forcing to explain
anything. CPU temperature does vary (33 → 39 °C) but at $\rho=-0.90$ with fan speed. §5 shows
that adding it changes the RMSE by under 0.5 % and gives it a **negative** coefficient, which
is the wrong sign for the mechanism it would supposedly capture — leakage power *increases*
with temperature (Liao, He & Lepak, *IEEE TCAD* 24(7), 2005). It is simply absorbing residual
fan-speed curvature. Out it goes.

## 3.5 The samples are not independent, and the statistics must say so

Samples arrive at 1 Hz inside plateaus that last three minutes. Consecutive residuals are
strongly autocorrelated (§4.4 measures lag-1 $\approx 0.73$, effective sample size ~400 rather
than ~2500). Two consequences shape everything below:

1. **Classical OLS standard errors are far too optimistic.** We use a *cluster (block)
   bootstrap* that resamples whole setpoints (Cameron & Miller, *J. Human Resources* 50(2),
   2015).
2. **Random $k$-fold cross-validation would be optimistically biased**, because a random split
   puts near-duplicate samples in both train and test. We use **leave-one-setpoint-out CV**:
   each fold holds out an entire fan speed the model has never seen. This is the blocked-CV
   prescription of Roberts et al. (*Ecography* 40(8), 2017), and it is the number to quote.

In [ ]:
# --- §3.3 made concrete: fit the UNRESTRICTED per-rotor cubic and look at what it claims.
#     P = P0 + sum_i ( b1i N_i + b2i N_i^2 + b3i N_i^3 )   -> 49 free parameters
X_free = np.column_stack([np.ones(int(fit_mask.sum())), K[fit_mask], K[fit_mask] ** 2, K[fit_mask] ** 3])
beta_free, *_ = np.linalg.lstsq(X_free, P[fit_mask], rcond=None)

x_hi = plateau_N[SWEEP_IDS[0]] / 1000.0
per_rotor_free = np.array([
    beta_free[1 + i] * x_hi[i] + beta_free[17 + i] * x_hi[i] ** 2 + beta_free[33 + i] * x_hi[i] ** 3
    for i in range(N_ROTORS)
])

print("power each rotor supposedly draws at 100 %, per the unrestricted fit [W]:")
print(np.round(per_rotor_free, 1))
print(f"\n  range {per_rotor_free.min():.0f} .. {per_rotor_free.max():.0f} W  "
      f"-> {int((per_rotor_free < 0).sum())} of {N_ROTORS} rotors are assigned NEGATIVE power")
print(f"  they happen to sum to {per_rotor_free.sum():.0f} W, which is roughly right —")
print("  the total is identified, the split across rotors is not.")

errs = []
for g in SWEEP_IDS:                                   # same leave-one-setpoint-out protocol
    te, tr_ = plateau_id == g, fit_mask & (plateau_id != g)
    Xa = np.column_stack([np.ones(int(tr_.sum())), K[tr_], K[tr_] ** 2, K[tr_] ** 3])
    Xb = np.column_stack([np.ones(int(te.sum())), K[te], K[te] ** 2, K[te] ** 3])
    ba, *_ = np.linalg.lstsq(Xa, P[tr_], rcond=None)
    errs.append(P[te] - Xb @ ba)
print(f"\n  in-sample RMSE {np.sqrt(((P[fit_mask] - X_free @ beta_free) ** 2).mean()):.2f} W with 49 parameters,")
print(f"  but leave-one-setpoint-out RMSE {np.sqrt((np.concatenate(errs) ** 2).mean()):.2f} W — worse than the")
print("  4-parameter pooled model. Fitting noise, exactly as predicted.")

That is the argument in one output: **negative fan power for a third of the rotors**, a total
that is nonetheless about right, and worse out-of-sample accuracy from twelve times as many
parameters. The pooled model that follows is not a simplification of the per-rotor model — it
is the only version of it this experiment can support.

In [ ]:
# --- estimation machinery
#
# Design matrix for  P = P0 + sum_i ( b1 N_i + b2 N_i^2 + ... + bD N_i^D )
# Because the same coefficients apply to every rotor, the model is linear in the
# "power sums"  S_d = sum_i (N_i / 1000)^d , one column per degree.

def design(deg: int, mask: np.ndarray, extra: list[np.ndarray] | None = None) -> np.ndarray:
    """Design matrix with an intercept and the power sums S_1..S_deg (rotor speed in kRPM)."""
    cols = [np.ones(int(mask.sum()))] + [(K[mask] ** d).sum(axis=1) for d in range(1, deg + 1)]
    if extra:
        cols += [e[mask] for e in extra]
    return np.column_stack(cols)


def design_pow(gamma: float, mask: np.ndarray) -> np.ndarray:
    """Design matrix for the single-term power law  P = P0 + k * sum_i N_i^gamma."""
    return np.column_stack([np.ones(int(mask.sum())), (K[mask] ** gamma).sum(axis=1)])


def ols(X: np.ndarray, y: np.ndarray) -> np.ndarray:
    beta, *_ = np.linalg.lstsq(X, y, rcond=None)
    return beta


def rmse(resid: np.ndarray) -> float:
    return float(np.sqrt(np.mean(resid ** 2)))


GAMMA_GRID = np.arange(1.5, 5.501, 0.005)


def fit_power_law(mask: np.ndarray) -> tuple[float, np.ndarray]:
    """Profile out gamma on a grid; for fixed gamma the model is linear, so this is exact."""
    best = None
    for g in GAMMA_GRID:
        X = design_pow(g, mask)
        beta = ols(X, P[mask])
        sse = float(((P[mask] - X @ beta) ** 2).sum())
        if best is None or sse < best[0]:
            best = (sse, g, beta)
    return best[1], best[2]


def grouped_cv(predict_from_fold, groups: np.ndarray = SWEEP_IDS) -> tuple[float, dict]:
    """
    Leave-one-setpoint-out CV. `predict_from_fold(train_mask, test_mask)` must return
    the predictions for the held-out setpoint, having seen only `train_mask`.
    """
    errs, per_group = [], {}
    for g in groups:
        test = plateau_id == g
        train = fit_mask & (plateau_id != g)
        e = P[test] - predict_from_fold(train, test)
        errs.append(e)
        per_group[int(g)] = rmse(e)
    return rmse(np.concatenate(errs)), per_group


print(f"fitting on {fit_mask.sum()} steady-state samples from {len(SWEEP_IDS)} setpoints "
      f"(pre-sweep block excluded: different PSU regime)")

## 4. Model selection

We compare, on exactly the same data:

* **poly D** — the additive polynomial of §3.2 with degrees $D = 1\ldots5$;
* **pure $N^{3}$** — the affinity law alone, $P = P_0 + k\sum_i N_i^{3}$, two parameters total;
* **free exponent** — $P = P_0 + k\sum_i N_i^{\gamma}$ with $\gamma$ estimated, which asks the
  data directly what the effective exponent is.

The decisive column is **CV RMSE**: leave-one-setpoint-out, so each number measures error at a
fan speed the model never saw. In-sample RMSE and $R^2$ are reported for completeness but
cannot discriminate between over- and well-fitted models here.

In [ ]:
rows, fitted = [], {}

for D in range(1, 6):
    X = design(D, fit_mask)
    beta = ols(X, P[fit_mask])
    r = P[fit_mask] - X @ beta
    cv, per_g = grouped_cv(
        lambda tr, te, D=D: design(D, te) @ ols(design(D, tr), P[tr])
    )
    fitted[f"poly D={D}"] = beta
    rows.append({"model": f"poly D={D}", "n_par": X.shape[1], "rmse_in_W": round(rmse(r), 3),
                 "r2": round(1 - r.var() / P[fit_mask].var(), 5),
                 "cv_rmse_W": round(cv, 3), "cv_worst_setpoint_W": round(max(per_g.values()), 2)})

X = design_pow(3.0, fit_mask)
beta_cubic = ols(X, P[fit_mask])
r = P[fit_mask] - X @ beta_cubic
cv, per_g = grouped_cv(lambda tr, te: design_pow(3.0, te) @ ols(design_pow(3.0, tr), P[tr]))
rows.append({"model": "pure N^3 (affinity)", "n_par": 2, "rmse_in_W": round(rmse(r), 3),
             "r2": round(1 - r.var() / P[fit_mask].var(), 5),
             "cv_rmse_W": round(cv, 3), "cv_worst_setpoint_W": round(max(per_g.values()), 2)})

gamma_hat, beta_pow = fit_power_law(fit_mask)
r = P[fit_mask] - design_pow(gamma_hat, fit_mask) @ beta_pow


def _power_law_fold(train: np.ndarray, test: np.ndarray) -> np.ndarray:
    g, b = fit_power_law(train)          # gamma is re-estimated inside every fold
    return design_pow(g, test) @ b


cv, per_g = grouped_cv(_power_law_fold)
rows.append({"model": f"free exponent (γ={gamma_hat:.2f})", "n_par": 3, "rmse_in_W": round(rmse(r), 3),
             "r2": round(1 - r.var() / P[fit_mask].var(), 5),
             "cv_rmse_W": round(cv, 3), "cv_worst_setpoint_W": round(max(per_g.values()), 2)})

selection_df = pl.DataFrame(rows)
with pl.Config(tbl_rows=20, tbl_width_chars=140):
    print(selection_df)

**Reading the table.**

* A **linear** model in speed is hopeless (CV RMSE ~13 W): the relationship is strongly convex,
  exactly as the affinity law predicts.
* The **affinity law on its own** — one fan parameter — already explains 99.2 % of the variance
  with a 3 W RMSE. That is the headline physical result: the cube law is essentially right.
* The **cubic polynomial ($D=3$)** is the best generaliser. Degrees 4 and 5 fit marginally
  better in-sample and *worse* out of sample: classic overfitting, and the extra terms are
  fitting the measurement quantisation, not physics.
* The **free exponent** lands at $\hat\gamma \approx 3.5$, i.e. slightly steeper than the ideal
  cube law, in the direction §3.2 predicts for a real motor (copper losses $\propto N^{4}$).

We adopt **poly $D=3$** as the primary model — it is the best out-of-sample performer, it nests
the affinity law, and it keeps the per-rotor interpretation intact. The quadratic is rejected
on physical grounds as well as statistical ones: it predicts *negative* fan power at the
minimum speed (§6).

In [ ]:
# --- Figure 4: model selection
fig, (ax_a, ax_b) = plt.subplots(1, 2, figsize=(9.5, 3.3), gridspec_kw={"wspace": 0.3})

# the linear model is off scale (see the table); showing it would compress everything that matters
keep = selection_df.with_row_index("i").filter(pl.col("model") != "poly D=1")
names = keep["model"].to_list()
cv_vals = keep["cv_rmse_W"].to_numpy()
in_vals = keep["rmse_in_W"].to_numpy()
best_i = int(np.argmin(cv_vals))

ypos = np.arange(len(names))
ax_a.barh(ypos, cv_vals, height=0.62, zorder=3,
          color=[BLUE if i == best_i else GRID for i in range(len(names))])
ax_a.plot(in_vals, ypos, "|", color=INK2, ms=11, mew=1.6, ls="none", zorder=5,
          label="in-sample RMSE")
ax_a.set_yticks(ypos, names, fontsize=8)
ax_a.invert_yaxis()
ax_a.set(xlabel="RMSE  [W]", xlim=(0, 5.4))
ax_a.set_title("a · leave-one-setpoint-out CV RMSE\n(linear model omitted: 12.9 W, off scale)")
ax_a.annotate(f"{cv_vals[best_i]:.2f} W", (cv_vals[best_i], ypos[best_i]),
              textcoords="offset points", xytext=(6, 0), va="center", fontsize=8.5,
              color=INK, fontweight="bold")
ax_a.legend(loc="lower right")
ax_a.grid(axis="y", visible=False)

# where each model errs, setpoint by setpoint
for label, D, color in (("poly D=2", 2, AQUA), ("poly D=3", 3, BLUE), ("poly D=4", 4, ORANGE)):
    _, per_g = grouped_cv(lambda tr, te, D=D: design(D, te) @ ols(design(D, tr), P[tr]))
    ax_b.plot(SWEEP_DUTY, [per_g[int(g)] for g in SWEEP_IDS], color=color, lw=1.8,
              marker="o", ms=4, label=label)
ax_b.set(xlabel="held-out setpoint  [%]", ylabel="CV RMSE at that setpoint  [W]")
ax_b.set_title("b · error is concentrated at the extrapolated endpoints")
ax_b.legend(loc="upper center")
ax_b.set_xticks(SWEEP_DUTY[::2])
ax_b.invert_xaxis()
save_fig(fig, "fig4-model-selection")
plt.show()

Panel *b* is worth a comment for the paper: the leave-one-out error is small and flat across
the interior of the sweep and spikes at 100 % and 25 %. Those two folds are **extrapolations**
— the model must predict beyond the convex hull of its training speeds — so they are the
strictest possible test, and the reason the quadratic's failure shows up there most clearly.

In [ ]:
# --- the primary model: additive cubic, m(0) = 0
DEG = 3
X_fit = design(DEG, fit_mask)
beta = ols(X_fit, P[fit_mask])
P0_hat, coef = beta[0], beta[1:]


def m_rotor(rpm: np.ndarray | float, c: np.ndarray = coef) -> np.ndarray | float:
    """Per-rotor power [W] as a function of that rotor's speed [RPM]. m(0) = 0 by construction."""
    x = np.asarray(rpm) / 1000.0
    return sum(c[d - 1] * x ** d for d in range(1, len(c) + 1))


def predict(rpm_matrix: np.ndarray) -> np.ndarray:
    """Node power [W] = baseline + sum of the 16 per-rotor contributions."""
    return P0_hat + m_rotor(rpm_matrix).sum(axis=1)


# cluster bootstrap over setpoints: the unit of resampling is a whole plateau
rng = np.random.default_rng(20260811)
N_BOOT = 2000
boot = np.empty((N_BOOT, DEG + 1))
idx_by_group = [np.flatnonzero(plateau_id == g) for g in SWEEP_IDS]
for b in range(N_BOOT):
    pick = rng.integers(0, len(SWEEP_IDS), len(SWEEP_IDS))
    idx = np.concatenate([idx_by_group[p] for p in pick])
    Xb = np.column_stack([np.ones(idx.size)] + [(K[idx] ** d).sum(axis=1) for d in range(1, DEG + 1)])
    boot[b] = ols(Xb, P[idx])
lo, hi = np.percentile(boot, [2.5, 97.5], axis=0)

print("P(t) = P0 + sum_i ( b1 N_i + b2 N_i^2 + b3 N_i^3 ),  N_i in kRPM,  P in W\n")
for name, est, l, h in zip(["P0", "b1", "b2", "b3"], beta, lo, hi):
    print(f"  {name:>3} = {est:>12.5f}   95 % cluster-bootstrap CI [{l:>10.5f}, {h:>10.5f}]")
print(f"\n  effective exponent from the free-exponent fit: γ = {gamma_hat:.2f}")
print(f"  m(N) is monotone increasing on [0, 21] kRPM: "
      f"{bool(np.all(np.diff(m_rotor(np.linspace(0, 21000, 500))) > 0))}")

## 4.2 Fitting the actual power, and the RMSE

The model is now applied to **every** sample in the run — including the spin-up/spin-down
ramps it was never fitted on, and the two held-out plateaus (the 100 % repeat and the
post-sweep automatic-control block).

In [ ]:
P_hat = predict(N)
resid = P - P_hat

eval_sets = [
    ("fit: 16 setpoints (steady state)", fit_mask),
    ("hold-out: 100 % repeat", plateau_id == REPEAT_ID),
    ("hold-out: post-sweep auto control", plateau_id == AUTO_ID),
    ("all plateaus (excl. pre-sweep)", plateau_id >= 1),
    ("full trace incl. ramps (t > 280 s)", t > 280),
    ("excluded: pre-sweep, 1-PSU regime", plateau_id == PRE_ID),
]
rows = []
for name, msk in eval_sets:
    rows.append({
        "evaluation set": name,
        "n": int(msk.sum()),
        "RMSE_W": round(rmse(resid[msk]), 3),
        "MAE_W": round(float(np.abs(resid[msk]).mean()), 3),
        "bias_W": round(float(resid[msk].mean()), 3),
        "MAPE_pct": round(float(100 * np.abs(resid[msk] / P[msk]).mean()), 3),
    })
cv_rmse = selection_df.filter(pl.col("model") == "poly D=3")["cv_rmse_W"].item()
rows.append({"evaluation set": "leave-one-setpoint-out CV", "n": int(fit_mask.sum()),
             "RMSE_W": cv_rmse, "MAE_W": None, "bias_W": None,
             "MAPE_pct": round(100 * cv_rmse / P[fit_mask].mean(), 3)})

error_df = pl.DataFrame(rows)
with pl.Config(tbl_rows=20, tbl_width_chars=150):
    print(error_df)

print(f"\nheadline: leave-one-setpoint-out RMSE = {cv_rmse:.2f} W "
      f"on a signal spanning {P[fit_mask].min():.0f}–{P[fit_mask].max():.0f} W "
      f"({100 * cv_rmse / P[fit_mask].mean():.2f} % of mean node power)")

In [ ]:
# --- Figure 5: the fan power law and the fitted curve
fig, (ax_a, ax_b) = plt.subplots(1, 2, figsize=(9.5, 3.6), gridspec_kw={"wspace": 0.28})

plateau_P = np.array([P[plateau_id == g].mean() for g in range(n_plateaus)])
grid_duty = np.linspace(0, 1, 200)
# sweep the whole rotor set proportionally between rest and the 100 % speeds
rotor_grid = grid_duty[:, None] * plateau_N[SWEEP_IDS[0]][None, :]
grid_mean = rotor_grid.mean(axis=1)

ax_a.plot(grid_mean / 1000, P0_hat + m_rotor(rotor_grid).sum(axis=1), color=BLUE, lw=2.0,
          label="additive cubic (D=3)", zorder=3)
ax_a.plot(grid_mean / 1000, beta_cubic[0] + beta_cubic[1] * ((rotor_grid / 1000) ** 3).sum(axis=1),
          color=ORANGE, lw=1.6, label="pure $N^3$ (affinity law)", zorder=2)
ax_a.plot(grid_mean / 1000, beta_pow[0] + beta_pow[1] * ((rotor_grid / 1000) ** gamma_hat).sum(axis=1),
          color=AQUA, lw=1.6, label=f"free exponent, γ={gamma_hat:.2f}", zorder=2)
ax_a.scatter(plateau_mean_rpm[SWEEP_IDS] / 1000, plateau_P[SWEEP_IDS], s=34, color=INK,
             zorder=5, lw=2, edgecolor=SURFACE, label="measured setpoint means")
ax_a.set(xlabel="mean rotor speed  [kRPM]", ylabel="node power  [W]", xlim=(0, 21), ylim=(195, 375))
ax_a.set_title("a · measured plateaus and the candidate laws")
ax_a.legend(loc="upper left")
ax_a.axvspan(0, plateau_mean_rpm[SWEEP_IDS].min() / 1000, color=GRID, alpha=0.5, lw=0, zorder=0)
ax_a.annotate("shaded: no data. The three laws are indistinguishable\nover the measured range, yet disagree by 37 W at $N=0$.",
              (2.9, 207), fontsize=7.6, color=MUTED, ha="left", va="center")

# How well-determined is the exponent? Profile the RMSE over gamma, with P0 concentrated
# out at each gamma. This needs no assumption about the (unidentified) baseline.
gammas = np.arange(2.0, 5.01, 0.02)
prof = np.array([rmse(P[fit_mask] - design_pow(g, fit_mask) @ ols(design_pow(g, fit_mask), P[fit_mask]))
                 for g in gammas])
ax_b.plot(gammas, prof, color=BLUE, lw=2.0, zorder=3)
ax_b.plot([gamma_hat], [prof.min()], "o", color=BLUE, ms=7, mec=SURFACE, mew=1.6, zorder=5)
rmse_at_3 = rmse(P[fit_mask] - design_pow(3.0, fit_mask) @ beta_cubic)
ax_b.plot([3.0], [rmse_at_3], "o", color=ORANGE, ms=7, mec=SURFACE, mew=1.6, zorder=5)
ax_b.annotate(f"affinity law\nγ = 3 → {rmse_at_3:.2f} W", (3.0, rmse_at_3),
              textcoords="offset points", xytext=(-6, 14), ha="right", fontsize=8, color=ORANGE)
ax_b.annotate(f"best fit\nγ = {gamma_hat:.2f} → {prof.min():.2f} W", (gamma_hat, prof.min()),
              textcoords="offset points", xytext=(14, 6), fontsize=8, color=BLUE)
ax_b.set(xlabel="exponent γ in  $P = P_0 + k\\sum_i N_i^{\\gamma}$", ylabel="RMSE  [W]")
ax_b.set_title("b · how well the data pins down the exponent")
save_fig(fig, "fig5-fan-law")
plt.show()

In [ ]:
# --- Figure 6: the model against the measured trace
fig, (ax_a, ax_b) = plt.subplots(2, 1, figsize=(9.5, 5.0), sharex=True,
                                 gridspec_kw={"height_ratios": [2.2, 1], "hspace": 0.14})

ax_a.plot(t / 60, P, color=INK2, lw=1.0, label="measured")
ax_a.plot(t / 60, P_hat, color=BLUE, lw=1.6, label="model: $P_0+\\sum_i m(N_i)$")
ax_a.axvspan(0, t[plateau_id == PRE_ID][-1] / 60, color=GRID, alpha=0.6, lw=0, zorder=0)
ax_a.annotate("excluded\n(1-PSU regime)", (t[plateau_id == PRE_ID].mean() / 60, 320),
              ha="center", fontsize=8, color=MUTED)
ax_a.set_ylabel("node power  [W]")
ax_a.set_title(f"Additive cubic fan model — CV RMSE {cv_rmse:.2f} W "
               f"({100 * cv_rmse / P[fit_mask].mean():.2f} % of mean node power)")
ax_a.legend(loc="upper right")

ax_b.axhline(0, color=AXIS, lw=0.9)
ax_b.plot(t / 60, resid, color=BLUE, lw=0.9)
ax_b.set(xlabel="elapsed time  [min]", ylabel="residual  [W]", ylim=(-20, 20))
ax_b.annotate("pre-sweep offset −13 W", (t[plateau_id == PRE_ID].mean() / 60, -13),
              textcoords="offset points", xytext=(6, -8), fontsize=8, color=CRITICAL)
save_fig(fig, "fig6-fit")
plt.show()

## 4.4 Residual diagnostics

Four checks, in the order that matters:

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(9.5, 6.0), gridspec_kw={"hspace": 0.36, "wspace": 0.26})
(ax1, ax2), (ax3, ax4) = axes
r_fit = resid[fit_mask]

ax1.axhline(0, color=AXIS, lw=0.9)
ax1.scatter(P_hat[fit_mask], r_fit, s=6, color=BLUE, alpha=0.35, lw=0)
ax1.set(xlabel="fitted power  [W]", ylabel="residual  [W]")
ax1.set_title("a · residual vs fitted")
ax1.annotate("100 % plateau: extra scatter\nwhile the PSUs settle", (364, 20),
             textcoords="offset points", xytext=(-10, 0), ha="right", fontsize=7.6, color=MUTED)

ax2.axhline(0, color=AXIS, lw=0.9)
ax2.scatter(t[fit_mask] / 60, r_fit, s=6, color=BLUE, alpha=0.35, lw=0)
ax2.axvline(1820 / 60, color=CRITICAL, lw=1.0)
ax2.annotate("+6 W step", (1820 / 60, 6.5), textcoords="offset points", xytext=(6, 0),
             fontsize=8, color=CRITICAL)
ax2.set(xlabel="elapsed time  [min]", ylabel="residual  [W]")
ax2.set_title("b · residual vs time")

ax3.axhline(0, color=AXIS, lw=0.9)
ax3.scatter(cpu_temp[fit_mask], r_fit, s=6, color=BLUE, alpha=0.35, lw=0)
ax3.set(xlabel="mean CPU die temperature  [°C]", ylabel="residual  [W]")
ax3.set_title("c · residual vs CPU temperature")

lags = np.arange(0, 181)
acf = np.array([1.0 if l == 0 else float(np.corrcoef(r_fit[:-l], r_fit[l:])[0, 1]) for l in lags])
ax4.axhline(0, color=AXIS, lw=0.9)
ax4.plot(lags, acf, color=BLUE, lw=1.6)
ax4.set(xlabel="lag  [s]", ylabel="residual autocorrelation")
ax4.set_title("d · residual autocorrelation")
n_eff = len(r_fit) * (1 - acf[1]) / (1 + acf[1])
ax4.annotate(f"lag-1 = {acf[1]:.2f}\n$n$ = {len(r_fit)}, $n_{{eff}}$ ≈ {n_eff:.0f}",
             (0.55, 0.75), xycoords="axes fraction", fontsize=8.5, color=INK2)
save_fig(fig, "fig7-residuals")
plt.show()

# --- what the structural break costs, and why we cannot simply regress it away
step = (t >= 1820).astype(float)
X_step = design(DEG, fit_mask, extra=[step])
beta_step = ols(X_step, P[fit_mask])
r_step = P[fit_mask] - X_step @ beta_step
S3 = (K[fit_mask] ** 3).sum(axis=1)

print("adding a step indicator at t = 1820 s:")
print(f"  step size           {beta_step[-1]:+.2f} W")
print(f"  in-sample RMSE      {rmse(r_fit):.3f} W  ->  {rmse(r_step):.3f} W")
print(f"  fan power at 100 %  {m_rotor(plateau_N[SWEEP_IDS[0]]).sum():.1f} W  ->  "
      f"{m_rotor(plateau_N[SWEEP_IDS[0]], beta_step[1:DEG + 1]).sum():.1f} W   <-- moves by 20 W!")
print(f"\n  corr(step indicator, sum N^3) = {np.corrcoef(step[fit_mask], S3)[0, 1]:+.3f}")
print(f"  corr(elapsed time, mean rotor speed) = {np.corrcoef(t[fit_mask], rpm_mean[fit_mask])[0, 1]:+.3f}")
print("\n  The sweep was run monotonically from 100 % down to 25 %, so elapsed time and fan speed")
print("  are confounded at rho = -0.998. A step indicator is therefore ~80 % collinear with the")
print("  fan term and cannot be separated from it: it 'explains' the break partly by stealing")
print("  20 W from the fan attribution. We keep the primary model WITHOUT it, and treat the")
print("  break as a known contaminant of the 60-55 % region rather than something we can adjust")
print("  away. See the design recommendation below.")

**(a)** No leftover curvature — the additive cubic has captured the shape. The residuals are
not homoscedastic, though: the 100 % plateau carries visibly more scatter (sd 6.1 W vs ~0.5 W
elsewhere) because it begins immediately after the redundant PSU wakes and the two supplies
spend ~30 s settling into load sharing. That inflates the RMSE slightly and is the main reason
the 100 % fold is the worst in panel *b* of the previous figure. A longer post-transition
settling window would remove it.

**(b)** One structural break, the +6 W step of §2.4. Adding an indicator for it drops the
in-sample RMSE from 2.02 to 1.68 W — but it also moves the fan power at 100 % by 20 W, because
the sweep was executed **monotonically in time** and elapsed time is therefore confounded with
fan speed at $\rho = -0.998$. Any background drift during the run is aliased with the treatment,
and no amount of post-hoc adjustment can undo that. We keep the model without the indicator and
flag the 60–55 % region as contaminated.

> **Design recommendation for the remaining experiments.** Run the setpoints in **randomised
> order**, and revisit two or three of them, rather than sweeping monotonically. Randomisation
> makes time-varying background drift orthogonal to the treatment instead of collinear with it,
> which is the only way a step like this becomes separable. Same protocol cost, strictly more
> information — this applies directly to the memstress and netstress runs still to come.

**(c)** Flat against CPU temperature — the assumption of §3.4 holds *ex post*.

**(d)** Lag-1 autocorrelation of 0.73 confirms that the 2485 samples are worth roughly 400
independent observations, which is why the CIs are cluster-bootstrapped and the CV is blocked.

## 5. Robustness: does temperature buy anything?

§3.4 argued on causal grounds that it should not be in the model. Here is the empirical
counterpart — adding each temperature signal to the primary model and reading off the change.

In [ ]:
rows = []
base = rmse(resid[fit_mask])
for name, extra in [("primary model (no temperature)", []),
                    ("+ CPU die temperature", [cpu_temp]),
                    ("+ board exhaust temperature", [exhaust_temp]),
                    ("+ CPU and exhaust", [cpu_temp, exhaust_temp])]:
    Xe = design(DEG, fit_mask, extra=extra)
    be = ols(Xe, P[fit_mask])
    re = P[fit_mask] - Xe @ be
    rows.append({"model": name, "rmse_W": round(rmse(re), 4),
                 "delta_rmse_pct": round(100 * (rmse(re) - base) / base, 2),
                 "temp_coef_W_per_K": None if not extra else round(float(be[DEG + 1]), 3)})
temp_df = pl.DataFrame(rows)
with pl.Config(tbl_width_chars=140):
    print(temp_df)
print(f"\ncorr(CPU temperature, mean rotor speed) = {np.corrcoef(cpu_temp[fit_mask], rpm_mean[fit_mask])[0, 1]:+.3f}")
print(f"inlet temperature over the whole run    = {np.unique(inlet_temp)} °C (no variation to exploit)")
print("\nThe CPU-temperature coefficient is negative — the wrong sign for leakage, which grows")
print("with temperature. It is absorbing residual fan-speed curvature, exactly the post-treatment")
print("bias of §3.4, and it buys < 0.5 % of RMSE. Temperature stays out of the model.")

## 6. The result: per-rotor power decomposition

This is the object the model was built for — a separate power figure for **every rotor**,
summing to the node's fan power.

In [ ]:
rows = []
for i, col in enumerate(FAN_COLS):
    rows.append({
        "rotor": col.replace("System_Board_", ""),
        "N_100pct_RPM": round(float(plateau_N[SWEEP_IDS[0], i])),
        "m_100pct_W": round(float(m_rotor(plateau_N[SWEEP_IDS[0], i])), 2),
        "N_25pct_RPM": round(float(plateau_N[SWEEP_IDS[-1], i])),
        "m_25pct_W": round(float(m_rotor(plateau_N[SWEEP_IDS[-1], i])), 2),
    })
rotor_power_df = pl.DataFrame(rows)
with pl.Config(tbl_rows=20):
    print(rotor_power_df)

fan_hi = float(m_rotor(plateau_N[SWEEP_IDS[0]]).sum())
fan_lo = float(m_rotor(plateau_N[SWEEP_IDS[-1]]).sum())
print(f"\ntotal fan power @100 % : {fan_hi:6.1f} W  ({100 * fan_hi / plateau_P[SWEEP_IDS[0]]:.0f} % of node power)")
print(f"total fan power @ 25 % : {fan_lo:6.1f} W  ({100 * fan_lo / plateau_P[SWEEP_IDS[-1]]:.0f} % of node power)")
print(f"per module (A+B) @100 %: {fan_hi / 8:6.2f} W")
print(f"measured swing 25 → 100 %: {plateau_P[SWEEP_IDS[0]] - plateau_P[SWEEP_IDS[-1]]:.1f} W "
      f"= {100 * (plateau_P[SWEEP_IDS[0]] / plateau_P[SWEEP_IDS[-1]] - 1):.0f} % more node power, "
      f"on an idle machine, for cooling alone")

## 7. What is *not* identified: the absolute baseline

This deserves to be stated plainly in the paper, because it is easy to quote $\hat P_0$ as if
it were measured.

The fans on this chassis **never stop**. The lowest speed reached — at the 25 % setpoint, which
turns out to coincide with the BMC's own automatic floor — is ~5.8 kRPM, about 32 % of maximum.
Splitting the measured power into "baseline" and "fan power at the floor" therefore requires
extrapolating $m(\cdot)$ from 5.8 kRPM down to 0, over a range where there is *no data at all*.

Models that are statistically indistinguishable on the observed range disagree by tens of watts
once extrapolated. The table below makes the size of that ambiguity explicit.

In [ ]:
def decompose(name: str, P0_m: float, coefs: np.ndarray | None = None,
              k: float | None = None, gamma: float | None = None) -> dict:
    """Fan power at the two ends of the sweep implied by a candidate model."""
    def fan(rpm_vec):
        x = rpm_vec / 1000.0
        if coefs is not None:
            return float(sum(coefs[d - 1] * (x ** d).sum() for d in range(1, len(coefs) + 1)))
        return float(k * (x ** gamma).sum())
    hi, lo_ = fan(plateau_N[SWEEP_IDS[0]]), fan(plateau_N[SWEEP_IDS[-1]])
    return {"model": name, "P0_W": round(float(P0_m), 1), "fan_100pct_W": round(hi, 1),
            "fan_25pct_W": round(lo_, 1), "fan_swing_W": round(hi - lo_, 1)}


baseline_df = pl.DataFrame([
    decompose("poly D=2", fitted["poly D=2"][0], coefs=fitted["poly D=2"][1:]),
    decompose("poly D=3 (primary)", fitted["poly D=3"][0], coefs=fitted["poly D=3"][1:]),
    decompose("poly D=4", fitted["poly D=4"][0], coefs=fitted["poly D=4"][1:]),
    decompose("pure N^3 (affinity)", beta_cubic[0], k=beta_cubic[1], gamma=3.0),
    decompose(f"free exponent γ={gamma_hat:.2f}", beta_pow[0], k=beta_pow[1], gamma=gamma_hat),
])
with pl.Config(tbl_width_chars=140):
    print(baseline_df)

# a model implying negative fan power at the minimum speed is physically inadmissible
ok = baseline_df.filter(pl.col("fan_25pct_W") > 0)
P0_LO, P0_HI = float(ok["P0_W"].min()), float(ok["P0_W"].max())
print(f"\nacross physically admissible models:")
print(f"  P0        spans {P0_LO:.0f} – {P0_HI:.0f} W    <- NOT identified, report as a range")
print(f"  fan swing spans {ok['fan_swing_W'].min():.1f} – {ok['fan_swing_W'].max():.1f} W  "
      f"<- robust: it is directly measured")
print("\nThe quadratic is rejected outright: it implies negative fan power at the minimum speed.")

**The honest statement for the paper.** What this experiment measures is the *difference*
in node power between fan speeds, and that quantity is robust across every admissible model:
going from the fan floor to 100 % costs **≈ 110 W** on an otherwise idle node. The *absolute*
attribution — how much of the 251 W drawn at the floor is fans and how much is the rest of the
machine — is a model-dependent extrapolation spanning ~45 W, and should be reported as a range.

Closing that gap needs an experiment that reaches lower speeds (or stops a fan): a per-fan PWM
path, or a physical ablation with one module pulled. That is the natural next measurement.

The figure below therefore anchors the decomposition to what was **measured** — the node power
at the fan floor — and shows the unidentified $P_0$ only as a band.

In [ ]:
# --- Figure 8: the decomposition
fig, (ax_a, ax_b) = plt.subplots(1, 2, figsize=(9.5, 3.6), gridspec_kw={"wspace": 0.3})

floor = plateau_P[SWEEP_IDS[-1]]
above = plateau_P[SWEEP_IDS] - floor
ax_a.bar(SWEEP_DUTY, np.full_like(above, floor), width=3.4, color=GRID,
         edgecolor=SURFACE, lw=2, zorder=3, label="node at minimum fan speed (measured)")
ax_a.bar(SWEEP_DUTY, above, bottom=floor, width=3.4, color=BLUE,
         edgecolor=SURFACE, lw=2, zorder=3, label="fan power above the floor (measured)")
# the unidentified baseline shown as a bracket in the right margin, never as a band across
# the bars — it is a property of the model, not a measured layer of the stack
ax_a.fill_betweenx([P0_LO, P0_HI], 20.5, 18.0, color=ORANGE, alpha=0.35, lw=0, zorder=3)
ax_a.annotate(f"$P_0$: {P0_LO:.0f}–{P0_HI:.0f} W\nnot identified (§7)", (19.2, P0_LO),
              textcoords="offset points", xytext=(0, -22), ha="center", fontsize=7.4, color=INK2)
for d, tot in ((100, plateau_P[SWEEP_IDS[0]]), (25, plateau_P[SWEEP_IDS[-1]])):
    ax_a.annotate(f"{tot:.0f} W", (d, tot), textcoords="offset points", xytext=(0, 5),
                  ha="center", fontsize=8, color=INK)
ax_a.set(xlabel="commanded duty cycle  [%]", ylabel="node power  [W]", ylim=(0, 420))
ax_a.set_title("a · what the fans cost, referenced to the floor")
ax_a.legend(loc="lower center", fontsize=7.4)
ax_a.set_xlim(105, 16)

grid_rpm = np.linspace(0, 21000, 300)
ax_b.plot(grid_rpm / 1000, m_rotor(grid_rpm), color=BLUE, lw=2.0, zorder=3)
ax_b.axvspan(0, plateau_N[SWEEP_IDS[-1]].min() / 1000, color=GRID, alpha=0.5, lw=0, zorder=0)
ax_b.scatter(plateau_N[SWEEP_IDS[0], is_a_rotor] / 1000, m_rotor(plateau_N[SWEEP_IDS[0], is_a_rotor]),
             s=34, color=BLUE, ec=SURFACE, lw=1.5, zorder=5, label="rotor A @ 100 %")
ax_b.scatter(plateau_N[SWEEP_IDS[0], ~is_a_rotor] / 1000, m_rotor(plateau_N[SWEEP_IDS[0], ~is_a_rotor]),
             s=34, color=ORANGE, ec=SURFACE, lw=1.5, zorder=5, label="rotor B @ 100 %")
ax_b.annotate("shaded: never\nobserved", (2.5, 11.0), fontsize=8, color=MUTED, ha="center")
ax_b.annotate("8 rotors per cluster,\nmarkers overlap", (17.6, 3.4), fontsize=7.6, color=MUTED, ha="center")
ax_b.set(xlabel="rotor speed  [kRPM]", ylabel="per-rotor power  $m(N)$  [W]")
ax_b.set_title("b · the fitted per-rotor law")
ax_b.legend(loc="upper left")
save_fig(fig, "fig8-decomposition")
plt.show()

## 8. Summary

| | |
|---|---|
| **Model** | $P = P_0 + \sum_{i=1}^{16}\left(\beta_1 N_i + \beta_2 N_i^2 + \beta_3 N_i^3\right)$, $N_i$ in kRPM |
| **Selection** | cubic beats degrees 1, 2, 4, 5 on leave-one-setpoint-out CV; degrees ≥ 4 overfit |
| **Accuracy** | CV RMSE ≈ **2.24 W** (0.78 % of mean node power) on a 250–401 W signal; $R^2 = 0.996$ |
| **Physics** | fitted effective exponent $\hat\gamma \approx 3.5$, consistent with the affinity law plus motor copper losses; the pure cube law alone already gives $R^2 = 0.992$ with one fan parameter |
| **Magnitude** | fans swing node power by ≈ 112 W (≈ 45 %) on an idle node; ≈ 19 W per fan module at 100 % |
| **Temperature** | correctly excluded — it is a *consequence* of fan speed here, and adds < 0.5 % of RMSE with the wrong sign |
| **Not identified** | the absolute baseline $P_0$ (206–251 W across equally good models), because the fans never stop |
| **Incidental** | PSU hot-spare is worth ≈ 14 W at idle; an unexplained +6 W background step at t ≈ 1820 s |

### Limitations

1. **One node, one run.** No between-node or between-day replication; the 100 % repeat gives a
   single repeatability estimate of ~3.6 W — which is *larger* than the model's CV RMSE, so
   at this point run-to-run reproducibility, not model form, is the accuracy ceiling.
2. **Rotors cannot be separated.** All fans share one command, so per-rotor *functions* are not
   estimable (§3.3); the model assumes identical rotors.
3. **The sweep was monotone in time**, confounding elapsed time with fan speed at
   $\rho=-0.998$ (§4.4). Randomise the setpoint order in the remaining experiments.
4. **Idle only.** $P_0$ was constant here by design. Under load, CPU power and fan speed move
   together and the decomposition needs the load term this notebook deliberately omits — that
   is what the memstress/netstress experiments are for.
5. **Fixed ambient.** Inlet temperature never moved (20 °C), so the model says nothing about
   ambient dependence — the very trade-off (El-Sayed et al. 2012) this model would be used to
   evaluate.
6. **BMC-grade instrumentation.** 1 Hz, 1 W quantisation, and unknown internal averaging in the
   PSU reading (Hackenberg et al., ISPASS 2013). Adequate here because we compare three-minute
   steady states, but not for transient work.
7. **One PSU-redundancy regime.** Everything above is fitted with both supplies active; the
   model does not transfer to a node running in hot-spare mode without the ~14 W offset of §2.4.

### References

**Fan aerodynamics and the affinity laws**
1. S. L. Dixon and C. A. Hall. *Fluid Mechanics and Thermodynamics of Turbomachinery*, 7th ed. Butterworth-Heinemann, 2014.
2. F. P. Bleier. *Fan Handbook: Selection, Application, and Design*. McGraw-Hill, 1997.
3. ASHRAE. *ASHRAE Handbook — HVAC Systems and Equipment*, ch. "Fans".
4. ANSI/AMCA Standard 210 / ASHRAE Standard 51, *Laboratory Methods of Testing Fans for Certified Aerodynamic Performance Rating*.

**Fan power and thermal management in servers**
5. Z. Wang, C. Bash, N. Tolia, M. Marwah, X. Zhu, P. Ranganathan. "Optimal Fan Speed Control for Thermal Management of Servers." *ASME InterPACK*, 2009.
6. N. El-Sayed, I. A. Stefanovici, G. Amvrosiadis, A. A. Hwang, B. Schroeder. "Temperature Management in Data Centers: Why Some (Might) Like It Hot." *ACM SIGMETRICS*, 2012.
7. M. K. Patterson. "The Effect of Data Center Temperature on Energy Efficiency." *ITHERM*, 2008.
8. ASHRAE TC 9.9. *Thermal Guidelines for Data Processing Environments*.

**Full-system power modelling**
9. D. Economou, S. Rivoire, C. Kozyrakis, P. Ranganathan. "Full-System Power Analysis and Modeling for Server Environments." *MoBS*, 2006.
10. S. Rivoire, P. Ranganathan, C. Kozyrakis. "A Comparison of High-Level Full-System Power Models." *HotPower*, 2008.
11. X. Fan, W.-D. Weber, L. A. Barroso. "Power Provisioning for a Warehouse-sized Computer." *ISCA*, 2007.
12. L. A. Barroso, U. Hölzle, P. Ranganathan. *The Datacenter as a Computer*, 3rd ed. Morgan & Claypool, 2018.

**Measurement**
13. D. Hackenberg, T. Ilsche, R. Schöne, D. Molka, M. Schmidt, W. E. Nagel. "Power Measurement Techniques on Standard Compute Nodes: A Quantitative Comparison." *IEEE ISPASS*, 2013.
14. W. Liao, L. He, K. M. Lepak. "Temperature and Supply Voltage Aware Performance and Power Modeling at Microarchitecture Level." *IEEE TCAD* 24(7), 2005.

**Statistical method**
15. D. A. Belsley, E. Kuh, R. E. Welsch. *Regression Diagnostics: Identifying Influential Data and Sources of Collinearity*. Wiley, 1980.
16. P. R. Rosenbaum. "The Consequences of Adjustment for a Concomitant Variable That Has Been Affected by the Treatment." *JRSS A* 147(5), 1984.
17. D. R. Roberts et al. "Cross-validation strategies for data with temporal, spatial, hierarchical, or phylogenetic structure." *Ecography* 40(8), 2017.
18. A. C. Cameron, D. L. Miller. "A Practitioner's Guide to Cluster-Robust Inference." *Journal of Human Resources* 50(2), 2015.

In [ ]:
# --- export the fitted model so the paper and downstream notebooks share one source of truth
model_card = {
    "node": USED_NODE,
    "window": {"start": T_START, "end": T_END},
    "form": "P_W = P0 + sum_i ( b1*n_i + b2*n_i**2 + b3*n_i**3 ),  n_i = rotor speed in kRPM",
    "rotors": FAN_COLS,
    "coefficients": {"P0_W": float(P0_hat),
                     **{f"b{d}": float(coef[d - 1]) for d in range(1, DEG + 1)}},
    "ci95_cluster_bootstrap": {name: [float(l), float(h)] for name, l, h
                               in zip(["P0_W", "b1", "b2", "b3"], lo, hi)},
    "validity_range_rpm": [float(N[fit_mask].min()), float(N[fit_mask].max())],
    "effective_exponent_gamma": float(gamma_hat),
    "errors_W": {"cv_leave_one_setpoint_out": float(cv_rmse),
                 "in_sample": float(rmse(resid[fit_mask])),
                 "all_plateaus": float(rmse(resid[plateau_id >= 1])),
                 "full_trace_with_ramps": float(rmse(resid[t > 280]))},
    "baseline_not_identified": {"note": "fans never stop; P0 is an extrapolation to N=0",
                                "range_W": [P0_LO, P0_HI]},
    "caveats": ["idle node only", "single PSU-redundancy regime (both supplies active)",
                "fans driven by a single duty cycle: per-rotor functions not identifiable",
                "pre-sweep block excluded (PSU hot-spare regime)"],
}
out_path = os.path.join(os.path.dirname(DATA_DIR), "fan_model.json")
with open(out_path, "w") as fh:
    json.dump(model_card, fh, indent=2)
print(f"wrote {out_path}")
print(json.dumps(model_card["coefficients"], indent=2))

---

# 9. Final check: model vs ground truth

Everything below uses the primary model — additive cubic, $D=3$ — applied to the **measured
rotor speeds** of every sample in the run, and compares it against the **measured node power**.

The error convention is

$$e(t) \;=\; P_\text{measured}(t) \;-\; P_\text{model}(t),$$

so a **negative** error means the model over-predicts and a **positive** error means it
under-predicts.

In [ ]:
# --- Figure 9: predicted vs ground truth
fig = plt.figure(figsize=(9.5, 6.4))
gs = fig.add_gridspec(2, 2, height_ratios=[1, 1.25], hspace=0.34, wspace=0.26)
ax_t = fig.add_subplot(gs[0, :])
ax_p = fig.add_subplot(gs[1, 0])
ax_h = fig.add_subplot(gs[1, 1])

# the three regimes the samples fall into
is_fit = fit_mask
is_hold = np.isin(plateau_id, [REPEAT_ID, AUTO_ID])
is_ramp = (plateau_id < 0) & (t > 280)
is_excl = plateau_id == PRE_ID

ax_t.axvspan(0, t[is_excl][-1] / 60, color=GRID, alpha=0.6, lw=0, zorder=0)
ax_t.plot(t / 60, P_hat, color=BLUE, lw=2.2, label="model prediction", zorder=2)
ax_t.plot(t / 60, P, color=INK, lw=0.8, label="ground truth (measured)", zorder=3)
ax_t.annotate("excluded\n(1-PSU regime)", (t[is_excl].mean() / 60, 330), ha="center",
              fontsize=8, color=MUTED)
ax_t.set(xlabel="elapsed time  [min]", ylabel="node power  [W]")
ax_t.set_title("a · predicted consumption against ground truth, whole run")
ax_t.legend(loc="upper right")

# parity plot: everything on one 1:1 scale
series = ((is_excl, MUTED, "excluded (1-PSU)"), (is_ramp, AQUA, "ramps / transients"),
          (is_fit, BLUE, "fitted setpoints"), (is_hold, ORANGE, "held-out plateaus"))
for msk, color, _ in series:                       # drawn back to front
    ax_p.scatter(P[msk], P_hat[msk], s=7, color=color, alpha=0.30, lw=0)
lims = [P.min() - 6, P.max() + 6]
ax_p.plot(lims, lims, color=INK, lw=1.0, zorder=5)
ax_p.annotate("1:1", (lims[1] - 6, lims[1] - 22), fontsize=8, color=INK2, ha="right")
ax_p.set(xlabel="measured power  [W]", ylabel="predicted power  [W]", xlim=lims, ylim=lims)
ax_p.set_aspect("equal")
ax_p.set_title("b · parity plot")
ax_p.legend(handles=[mpl.lines.Line2D([], [], ls="none", marker="o", ms=5, color=c, label=l)
                     for _, c, l in reversed(series)], loc="upper left", fontsize=7.6)

# error distribution on the sets the model is meant to cover
err_main = resid[plateau_id >= 1]
CLIP_LO, CLIP_HI = -11.0, 12.0
ax_h.hist(np.clip(err_main, CLIP_LO, CLIP_HI), bins=np.linspace(CLIP_LO, CLIP_HI, 70),
          color=BLUE, lw=0)
ax_h.axvline(0, color=INK, lw=1.0)
ax_h.axvline(float(np.median(err_main)), color=ORANGE, lw=1.4)
ax_h.annotate(f"median {np.median(err_main):+.2f} W", (float(np.median(err_main)), 0.92),
              xycoords=("data", "axes fraction"), textcoords="offset points", xytext=(8, 0),
              fontsize=8, color=ORANGE)
n_over = int((err_main > CLIP_HI).sum())
ax_h.annotate(f"{n_over} PSU-transient samples\nreach +{err_main.max():.0f} W (piled at the edge)",
              (CLIP_HI, 0.60), xycoords=("data", "axes fraction"), textcoords="offset points",
              xytext=(-6, 0), ha="right", fontsize=7.4, color=MUTED)
ax_h.set(xlabel="error  $P_{measured}-P_{model}$  [W]", ylabel="samples", xlim=(CLIP_LO, CLIP_HI))
ax_h.set_title("c · error distribution, all plateaus")
save_fig(fig, "fig9-prediction-vs-truth")
plt.show()

In [ ]:
# --- the requested error table (errors are SIGNED: measured - predicted)
report_sets = [
    ("fit: 16 setpoints", fit_mask),
    ("all plateaus (excl. pre-sweep)", plateau_id >= 1),
    ("full trace incl. ramps", t > 280),
]

metrics = {"RMSE [W]": lambda e: np.sqrt(np.mean(e ** 2)),
           "max(error) [W]": np.max,
           "median(error) [W]": np.median,
           "min(error) [W]": np.min}

error_table = pl.DataFrame(
    [{"metric": name, **{lbl: round(float(fn(resid[msk])), 3) for lbl, msk in report_sets}}
     for name, fn in metrics.items()]
)
with pl.Config(tbl_width_chars=140):
    print(error_table)

print("\nerror = measured - predicted; negative => the model over-predicts.")

main = plateau_id >= 1
i_hi, i_lo = np.flatnonzero(main)[resid[main].argmax()], np.flatnonzero(main)[resid[main].argmin()]
print(f"both extremes fall in the same 30 s window of the first 100 % plateau:")
print(f"  max {resid[i_hi]:+6.1f} W at t = {t[i_hi]:.0f} s   (plateau {plateau_id[i_hi]})")
print(f"  min {resid[i_lo]:+6.1f} W at t = {t[i_lo]:.0f} s   (plateau {plateau_id[i_lo]})")
print("That is the PSU load-sharing transient of §2.4 ringing right after the redundant supply")
print("wakes, not a failure of the fan law: the median error is ~0 W and the interquartile range")
print(f"is [{np.percentile(resid[main], 25):+.2f}, {np.percentile(resid[main], 75):+.2f}] W.")
print(f"Excluding that first plateau, the error range narrows to "
      f"[{resid[plateau_id >= 2].min():+.2f}, {resid[plateau_id >= 2].max():+.2f}] W "
      f"with RMSE {rmse(resid[plateau_id >= 2]):.2f} W.")
print(f"\nThe largest systematic (not transient) miss is the 100 % repeat, over-predicted by "
      f"{abs(resid[plateau_id == REPEAT_ID].mean()):.1f} W —")
print("that is the run-to-run repeatability of the rig, and it exceeds the model's own CV RMSE.")

---

# 10. The same analysis under two different assumptions

Everything up to here treated the run as what it physically is: a time series of 16 dependent
three-minute experiments. This section repeats the analysis end to end under two changes
requested for comparison:

1. **Every sample is treated as an independent observation** — the timestamp column is
   discarded, and with it the plateau structure. Random $k$-fold cross-validation replaces
   leave-one-setpoint-out; classical OLS standard errors replace the cluster bootstrap.
2. **The data are cropped to start at the 95 % setpoint**, so the *first* 100 % block is
   dropped and the **100 % repeat** near the end supplies the top of the range. Everything from
   the 100 %→95 % transition to the end of the recording is kept.

The crop is a clear improvement and I would keep it: it removes both contaminated regions in
one cut — the single-PSU pre-sweep block of §2.4 and the noisy first 100 % plateau where the
supplies were still settling (the source of every large error in §9). The 100 % repeat is the
cleaner measurement of the same condition (sd 0.58 W vs 6.05 W).

The independence assumption is the part to watch. Section 3.5 showed the lag-1 residual
autocorrelation is 0.73, so the ~3 500 rows carry roughly 500 independent observations' worth
of information. Asserting independence does not create the missing information — it hides the
fact that it is missing. Below, rather than argue the point, the two protocols are simply run
side by side so the consequences are visible in the numbers.

One thing does **not** change: the identifiability result of §3.3. That the 16 rotors share a
single command is a property of the experiment, not of the sampling assumption or the crop.

In [ ]:
# --- the crop: everything from the 100 % -> 95 % transition onward
crop = t > t[plateau_id == SWEEP_IDS[0]][-1]
crop_steady = crop & (plateau_id >= 0)

print(f"crop starts at t = {t[crop][0]:.0f} s")
print(f"  rows kept          : {crop.sum()} of {len(t)}  ({100 * crop.mean():.0f} %)")
print(f"  of which steady     : {crop_steady.sum()}   ramps/transients: {(crop & ~crop_steady).sum()}")
print(f"  plateaus in the crop: {sorted(set(plateau_id[crop_steady].tolist()))}")
print(f"  power range         : {P[crop].min():.0f} – {P[crop].max():.0f} W")
print(f"  dropped             : pre-sweep (1-PSU) and the first 100 % plateau")


def kfold_cv(build, mask: np.ndarray, n_folds: int = 10, seed: int = 7) -> float:
    """Random k-fold CV — the protocol that is correct only if rows are independent."""
    idx = np.flatnonzero(mask)
    np.random.default_rng(seed).shuffle(idx)
    errs = []
    for fold in np.array_split(idx, n_folds):
        test = np.zeros(len(t), dtype=bool)
        test[fold] = True
        errs.append(P[test] - build(mask & ~test, test))
    return rmse(np.concatenate(errs))

In [ ]:
# --- model selection under the i.i.d. protocol
rows = []
for D in range(1, 6):
    X = design(D, crop)
    b = ols(X, P[crop])
    r = P[crop] - X @ b
    n_c, p_c = int(crop.sum()), X.shape[1]
    rss = float((r ** 2).sum())
    rows.append({
        "model": f"poly D={D}", "n_par": p_c,
        "rmse_in_W": round(rmse(r), 3),
        "cv10_rmse_W": round(kfold_cv(lambda tr, te, D=D: design(D, te) @ ols(design(D, tr), P[tr]), crop), 3),
        "BIC": round(n_c * np.log(rss / n_c) + p_c * np.log(n_c), 1),
    })
X = design_pow(3.0, crop)
b = ols(X, P[crop])
rows.append({"model": "pure N^3 (affinity)", "n_par": 2, "rmse_in_W": round(rmse(P[crop] - X @ b), 3),
             "cv10_rmse_W": round(kfold_cv(lambda tr, te: design_pow(3.0, te) @ ols(design_pow(3.0, tr), P[tr]), crop), 3),
             "BIC": None})
gamma_crop, beta_pow_crop = fit_power_law(crop)
rows.append({"model": f"free exponent (γ={gamma_crop:.2f})", "n_par": 3,
             "rmse_in_W": round(rmse(P[crop] - design_pow(gamma_crop, crop) @ beta_pow_crop), 3),
             "cv10_rmse_W": round(kfold_cv(_power_law_fold, crop), 3), "BIC": None})

selection_iid_df = pl.DataFrame(rows)
with pl.Config(tbl_rows=20, tbl_width_chars=140):
    print(selection_iid_df)

poly_rows = selection_iid_df.filter(pl.col("BIC").is_not_null())
gap = (poly_rows["cv10_rmse_W"] - poly_rows["rmse_in_W"]).abs().max()
print(f"\nlargest gap between 10-fold CV and in-sample RMSE across the polynomials: {gap:.3f} W")
print("Random k-fold has stopped measuring generalisation. Each held-out sample has a neighbour")
print("one second away in the training fold, so CV simply reproduces the in-sample fit and can")
print("no longer penalise complexity — CV and BIC both keep improving with degree, without limit.")

In [ ]:
# --- what the physical constraint says about the degrees that CV and BIC now prefer
rows = []
N_hi, N_lo = plateau_N[REPEAT_ID], plateau_N[SWEEP_IDS[-1]]
for D in range(2, 6):
    b = ols(design(D, crop), P[crop])
    fan = lambda x, b=b, D=D: float(sum(b[d] * ((x / 1000.0) ** d).sum() for d in range(1, D + 1)))
    rows.append({"model": f"poly D={D}", "P0_W": round(float(b[0]), 1),
                 "fan_100pct_W": round(fan(N_hi), 1), "fan_25pct_W": round(fan(N_lo), 1),
                 "physically_admissible": bool(fan(N_lo) > 0 and fan(N_hi) > fan(N_lo))})
with pl.Config(tbl_width_chars=140):
    print(pl.DataFrame(rows))
print("\nD=5 — the degree favoured by both CV and BIC here — implies a 521 W baseline and")
print("NEGATIVE fan power. With the timestamp discarded, physics is the only guardrail left,")
print("so we keep D=3: the same functional form as §4, chosen on grounds the data can still support.")

In [ ]:
# --- the D=3 model on the cropped data, with i.i.d. standard errors
X_crop = design(3, crop)
beta_crop = ols(X_crop, P[crop])
resid_crop_fit = P[crop] - X_crop @ beta_crop
n_c, p_c = int(crop.sum()), X_crop.shape[1]
sigma2 = float((resid_crop_fit ** 2).sum() / (n_c - p_c))
se_iid = np.sqrt(np.diag(sigma2 * np.linalg.inv(X_crop.T @ X_crop)))

print("P = P0 + sum_i ( b1 N_i + b2 N_i^2 + b3 N_i^3 ),  N_i in kRPM   [cropped, i.i.d.]\n")
print(f"{'':>4} {'estimate':>12} {'SE (iid)':>10} {'95 % CI (iid)':>26} {'width vs §4 cluster CI':>24}")
for k_, est, s in zip(["P0", "b1", "b2", "b3"], beta_crop, se_iid):
    w_iid = 2 * 1.96 * s
    j = ["P0", "b1", "b2", "b3"].index(k_)
    w_blk = float(hi[j] - lo[j])
    print(f"{k_:>4} {est:>12.5f} {s:>10.5f}   [{est - 1.96 * s:>10.5f}, {est + 1.96 * s:>9.5f}]"
          f"   {w_blk / w_iid:>19.1f}x wider")
print(f"\nThe blocked cluster-bootstrap intervals of §4 are ~12x wider on every coefficient.")
print("Both cannot be right: the i.i.d. intervals count each of ~3500 correlated seconds as")
print("fresh evidence, when the autocorrelation of §4.4 says they are worth roughly 500.")


def m_rotor_crop(rpm):
    """Per-rotor power [W] under the cropped i.i.d. fit."""
    x = np.asarray(rpm) / 1000.0
    return sum(beta_crop[d] * x ** d for d in range(1, 4))


P_hat_crop = beta_crop[0] + m_rotor_crop(N).sum(axis=1)
resid_crop = P - P_hat_crop

fan_hi_c = float(m_rotor_crop(N_hi).sum())
fan_lo_c = float(m_rotor_crop(N_lo).sum())
print(f"\nP0 = {beta_crop[0]:.1f} W | fan @100 % = {fan_hi_c:.1f} W | fan @25 % = {fan_lo_c:.1f} W "
      f"| swing = {fan_hi_c - fan_lo_c:.1f} W | per module @100 % = {fan_hi_c / 8:.2f} W")

In [ ]:
# --- error table for the cropped i.i.d. model (errors are signed: measured - predicted)
report_sets_c = [
    ("all cropped rows", crop),
    ("cropped, steady state only", crop_steady),
    ("cropped, ramps only", crop & ~crop_steady),
]
error_table_iid = pl.DataFrame(
    [{"metric": name, **{lbl: round(float(fn(resid_crop[msk])), 3) for lbl, msk in report_sets_c}}
     for name, fn in metrics.items()]
)
with pl.Config(tbl_width_chars=140):
    print(error_table_iid)

ramp_share = float((resid_crop[crop & ~crop_steady] ** 2).sum() / (resid_crop[crop] ** 2).sum())
print(f"\nramps are {100 * (crop & ~crop_steady).sum() / crop.sum():.0f} % of the cropped rows "
      f"but {100 * ramp_share:.0f} % of the squared error.")
i_hi = np.flatnonzero(crop)[resid_crop[crop].argmax()]
i_lo = np.flatnonzero(crop)[resid_crop[crop].argmin()]
print(f"both extremes are spin transitions, not setpoints: "
      f"max {resid_crop[i_hi]:+.1f} W at t = {t[i_hi]:.0f} s, min {resid_crop[i_lo]:+.1f} W at t = {t[i_lo]:.0f} s")
print("During a ramp the tachometer and the power meter are not synchronised, so the model is")
print("asked to predict a transient from a lagged input. Discarding the timestamp is exactly what")
print("makes these rows indistinguishable from steady-state ones.")

In [ ]:
# --- Figure 10: cropped i.i.d. model vs ground truth
fig = plt.figure(figsize=(9.5, 6.4))
gs = fig.add_gridspec(2, 2, height_ratios=[1, 1.25], hspace=0.34, wspace=0.26)
ax_t = fig.add_subplot(gs[0, :])
ax_p = fig.add_subplot(gs[1, 0])
ax_h = fig.add_subplot(gs[1, 1])

ax_t.axvspan(0, t[crop][0] / 60, color=GRID, alpha=0.6, lw=0, zorder=0)
ax_t.plot(t / 60, P_hat_crop, color=BLUE, lw=2.2, label="model prediction (cropped, i.i.d.)", zorder=2)
ax_t.plot(t / 60, P, color=INK, lw=0.8, label="ground truth (measured)", zorder=3)
ax_t.annotate("cropped away", (t[crop][0] / 120, 330), ha="center", fontsize=8, color=MUTED)
ax_t.set(xlabel="elapsed time  [min]", ylabel="node power  [W]")
ax_t.set_title("a · cropped i.i.d. model against ground truth")
ax_t.legend(loc="upper right")

series_c = ((~crop, MUTED, "cropped away"), (crop & ~crop_steady, AQUA, "ramps / transients"),
            (crop_steady, BLUE, "steady state (cropped)"))
for msk, color, _ in series_c:
    ax_p.scatter(P[msk], P_hat_crop[msk], s=7, color=color, alpha=0.30, lw=0)
lims = [P.min() - 6, P.max() + 6]
ax_p.plot(lims, lims, color=INK, lw=1.0, zorder=5)
ax_p.annotate("1:1", (lims[1] - 6, lims[1] - 22), fontsize=8, color=INK2, ha="right")
ax_p.set(xlabel="measured power  [W]", ylabel="predicted power  [W]", xlim=lims, ylim=lims)
ax_p.set_aspect("equal")
ax_p.set_title("b · parity plot")
ax_p.legend(handles=[mpl.lines.Line2D([], [], ls="none", marker="o", ms=5, color=c, label=l)
                     for _, c, l in reversed(series_c)], loc="upper left", fontsize=7.6)

err_c = resid_crop[crop]
ax_h.hist(np.clip(err_c, CLIP_LO, CLIP_HI), bins=np.linspace(CLIP_LO, CLIP_HI, 70), color=BLUE, lw=0)
ax_h.axvline(0, color=INK, lw=1.0)
ax_h.axvline(float(np.median(err_c)), color=ORANGE, lw=1.4)
ax_h.annotate(f"median {np.median(err_c):+.2f} W", (float(np.median(err_c)), 0.92),
              xycoords=("data", "axes fraction"), textcoords="offset points", xytext=(8, 0),
              fontsize=8, color=ORANGE)
ax_h.annotate(f"{int((err_c > CLIP_HI).sum())} ramp samples reach\n+{err_c.max():.0f} W (piled at the edge)",
              (CLIP_HI, 0.60), xycoords=("data", "axes fraction"), textcoords="offset points",
              xytext=(-6, 0), ha="right", fontsize=7.4, color=MUTED)
ax_h.set(xlabel="error  $P_{measured}-P_{model}$  [W]", ylabel="samples", xlim=(CLIP_LO, CLIP_HI))
ax_h.set_title("c · error distribution, all cropped rows")
save_fig(fig, "fig10-iid-cropped")
plt.show()

In [ ]:
# --- the two protocols side by side
comparison_df = pl.DataFrame([
    {"quantity": "rows used in the fit", "§1-9  blocked, full sweep": f"{int(fit_mask.sum())}",
     "§10  i.i.d., cropped": f"{int(crop.sum())}"},
    {"quantity": "source of the 100 % point", "§1-9  blocked, full sweep": "first 100 % block (sd 6.05 W)",
     "§10  i.i.d., cropped": "100 % repeat (sd 0.58 W)"},
    {"quantity": "cross-validation", "§1-9  blocked, full sweep": "leave-one-setpoint-out",
     "§10  i.i.d., cropped": "random 10-fold"},
    {"quantity": "CV RMSE [W]", "§1-9  blocked, full sweep": f"{cv_rmse:.2f}",
     "§10  i.i.d., cropped": f"{selection_iid_df.filter(pl.col('model') == 'poly D=3')['cv10_rmse_W'].item():.2f}"},
    {"quantity": "does CV discriminate?", "§1-9  blocked, full sweep": "yes — D=3 wins, D>=4 overfits",
     "§10  i.i.d., cropped": "no — improves with degree without limit"},
    {"quantity": "P0 [W]", "§1-9  blocked, full sweep": f"{P0_hat:.1f}",
     "§10  i.i.d., cropped": f"{beta_crop[0]:.1f}"},
    {"quantity": "fan power @100 % [W]", "§1-9  blocked, full sweep": f"{m_rotor(plateau_N[SWEEP_IDS[0]]).sum():.1f}",
     "§10  i.i.d., cropped": f"{fan_hi_c:.1f}"},
    {"quantity": "fan swing 25->100 % [W]", "§1-9  blocked, full sweep": f"{m_rotor(plateau_N[SWEEP_IDS[0]]).sum() - m_rotor(plateau_N[SWEEP_IDS[-1]]).sum():.1f}",
     "§10  i.i.d., cropped": f"{fan_hi_c - fan_lo_c:.1f}"},
    {"quantity": "effective exponent γ", "§1-9  blocked, full sweep": f"{gamma_hat:.2f}",
     "§10  i.i.d., cropped": f"{gamma_crop:.2f}"},
    {"quantity": "width of 95 % CI on b3", "§1-9  blocked, full sweep": f"{hi[3] - lo[3]:.5f}",
     "§10  i.i.d., cropped": f"{2 * 1.96 * se_iid[3]:.5f}"},
])
with pl.Config(tbl_width_chars=170, fmt_str_lengths=60):
    print(comparison_df)

## 10.1 What the comparison shows

**The physics is stable.** The fan swing between 25 % and 100 % moves by ~3 W between the two
protocols (113.3 → 110.1 W) and the effective exponent by 0.14 (3.54 → 3.40). The headline
results of this notebook — a cubic-dominated additive law, fans worth ~110 W on an idle node,
~19 W per module at full speed — do not depend on either choice. That is reassuring, and it is
the honest way to report robustness.

**The crop is a genuine improvement** and should be kept for the paper. It removes the
single-PSU block and the noisy first 100 % plateau in one cut, and it puts the cleaner repeat
measurement at the top of the range. Every large error in §9 came from the region it deletes.

**The independence assumption costs the two things the model needs most.** With the timestamp
discarded:

* *Model selection stops working.* Random 10-fold CV lands within 0.003 W of the in-sample
  RMSE, because every held-out second has a near-duplicate one second away in the training
  folds. CV and BIC then both improve monotonically with polynomial degree, and the degree they
  favour, $D=5$, implies a 521 W baseline and negative fan power. Physical admissibility, not
  statistics, is what rejects it.
* *Uncertainty is understated.* The i.i.d. intervals are ~12x narrower than the
  cluster-bootstrap intervals over the same data — consistent with an effective sample size
  near 500 rather than 3 500. Nothing was learned in between; the second set simply counts
  each correlated second as fresh evidence.

**Ramps become invisible.** Without the timestamp there is no way to tell a spin transition from
a steady state, so all of them enter the fit. They are 16 % of the rows and 77 % of the squared
error, and both extreme errors in the table (+66 W, −19 W) are spin transitions where the
tachometer and the power meter are momentarily out of step. Restricting to steady state within
the same crop halves the RMSE, from 2.62 W to 1.37 W.

**Recommendation.** Adopt the crop; keep the blocked treatment. Concretely: fit on the
steady-state rows from the 95 % setpoint onward, select and quote uncertainty with
leave-one-setpoint-out CV and the cluster bootstrap. That combination is the best of both —
it is the configuration with the lowest error in this notebook (RMSE 1.37 W) *and* the only one
whose model selection and error bars mean what they say.